# **Quantum Computing — the Practical Way**
### *QPlayLearn*

## **Installation**

First, we install the packages we need in the current environment. They will not be installed on your local machine.

In [ ]:
# Qiskit is the open-source library for quantum computing founded by IBM
! pip install qiskit qiskit-aer qiskit-ibm-runtime 
! pip install matplotlib pylatexenc

## **Importing packages**

We import all the packages we are going to need to run the code. 
Remember to run this cell before every Sandbox!

In [ ]:
import qiskit as qk
from qiskit.quantum_info import Statevector # to get the state coefficients
from qiskit_aer import AerSimulator # to run circuits on the quantum computer simulator
from qiskit.visualization import plot_bloch_multivector, plot_bloch_vector, plot_histogram

# Packages for graphical representations and plots
import matplotlib as mpl
import matplotlib.pyplot as plt

# Math library
import numpy as np

## **SANDBOX — Quantum teleportation**

In superdense coding, Alice sent one qubit so that Bob could recover two **classical bits**. Quantum teleportation turns this pattern around: Alice sends two classical bits so that Bob can recover a **quantum state**.

This time we will use three qubits and conditional operations, building the circuit one stage at a time. We’ll guide you throuugh the steps of the algorithm, but the actual code is on you! Decide if you want to write it cell by cell, or all together at the end



### **Meet the three qubits**

- $q_2$ contains the state $|\psi\rangle$ that Alice wants to teleport.
- $q_1$ is Alice’s half of an entangled pair.
- $q_0$ is Bob’s half of the pair, the qubit that will receive $|\psi\rangle$.

We also have two classical bits for Alice's measurement outcomes, and one classical bit for Bob's final measurement.

In [ ]:
# Three qubits, two bits for Alice's measurements and one bit for Bob's final check
q = qk.QuantumRegister(3, "q")
alice_bits = qk.ClassicalRegister(2, "alice")
check = qk.ClassicalRegister(1, "check")
qc = qk.QuantumCircuit(q, alice_bits, check)

### **1 — Prepare the state to teleport**

Alice encodes the state $|\psi\rangle$ she wants to teleport into qubit $q_2$. An unknown qubit state can be written as

$$|\psi\rangle=a|0\rangle+b|1\rangle.$$

To test the protocol, we will prepare a specific state using two rotations. We know how it was prepared, but Bob doesn't. In other words, the teleportation circuit will not use this information. Try changing `theta` and `phi` later to test a different state.

In [ ]:
# Choose a state |psi> and prepare it on Alice's message qubit q2
theta = 1.1
phi = 0.7

qc.ry(theta, q[2])
qc.rz(phi, q[2])

qc.draw(output="mpl")

### **2 — Alice and Bob share entanglement**

Alice and Bob prepare a Bell pair,

$$|\Phi^+\rangle_{q_1q_0}=\frac{|00\rangle+|11\rangle}{\sqrt{2}}.$$

Alice keeps $q_1$, while Bob holds $q_0$. This pair must be shared before the protocol begins.

In [ ]:
# Your turn: create a Bell pair between Alice’s q1 and Bob’s q0

# qc.h(...)
# qc.cx(..., ...)

qc.barrier()
qc.draw(output="mpl")

At this point, the three-qubit state is

$$|\psi\rangle_{q_2}|\Phi^+\rangle_{q_1q_0}.$$

### **3 — Alice performs the teleportation gates**

Alice first applies a CNOT on qubit $q_2$ (control) and qubit $q_1$ (target). She then applies a Hadamard gate on $q_2$.

In [ ]:
# Your turn: apply a CNOT from q2 to q1
# qc.cx(..., ...)


# Your turn: change the basis of q2 with a Hadamard gate
# qc.h(...)

qc.draw(output="mpl")

The terms in the resulting state can be grouped according to Alice’s qubits $q_2q_1, obtaining a useful expression

$$\begin{aligned}
\frac{1}{2}\big[&|00\rangle_{q_2q_1}(a|0\rangle+b|1\rangle)_{q_0}\\
+&|01\rangle_{q_2q_1}(a|1\rangle+b|0\rangle)_{q_0}\\
+&|10\rangle_{q_2q_1}(a|0\rangle-b|1\rangle)_{q_0}\\
+&|11\rangle_{q_2q_1}(a|1\rangle-b|0\rangle)_{q_0}\big].
\end{aligned}$$
This compact expression contains the logic of the protocol: Alice’s measurement outcome tells Bob how his qubit differs from $|\psi\rangle=a|0\rangle+b|1\rangle$.

<details>
<summary><strong>How did we get this expression?</strong></summary>

Starting from

$$|\psi\rangle_{q_2}|\Phi^+\rangle_{q_1q_0}=\frac{1}{\sqrt{2}}\left[a(|000\rangle+|011\rangle)+b(|100\rangle+|111\rangle)\right],$$

the CNOT gives (remember that the qubits are ordered as $q_2q_1q_0$)

$$\frac{1}{\sqrt{2}}\left[a(|000\rangle+|011\rangle)+b(|110\rangle+|101\rangle)\right],$$

The Hadamard redistributes both the $a$ and $b$ terms between the states $|0\rangle$ and $|1\rangle$ of $q_2$.

Remember that

$$H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}},\qquad H|1\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.$$

Applying $H_{q_2}$ to the state after the CNOT gives

$$\frac{1}{2}\left[a(|000\rangle+|100\rangle+|011\rangle+|111\rangle)+b(|010\rangle-|110\rangle+|001\rangle-|101\rangle)\right].$$

Regrouping these terms by the first two qubits, which Alice is about to measure, we obtain the expression above.

Equivalently, the four states on Bob’s qubit are $|\psi\rangle$, $X|\psi\rangle$, $Z|\psi\rangle$ and $XZ|\psi\rangle$.

</details>

### **4 — Alice measures and sends two classical bits**

Alice measures $q_1$ and $q_2$. Each pair of outcomes—`00`, `01`, `10` or `11`—occurs with equal probability.

The measurement destroys the original state on $q_2$. However, the information needed to reconstruct it is now distributed between Bob’s qubit and Alice’s two classical outcomes.

In [ ]:
# Store the outcome from q1 in alice[0] and from q2 in alice[1]
qc.measure(q[1], alice_bits[0])
qc.measure(q[2], alice_bits[1])

qc.draw(output="mpl")

### **5 — Bob reconstructs the state**

Alice sends her two measurement outcomes to Bob through a classical channel. As the expression above shows, they tell him which correction to apply:

| Alice’s outcome | State of Bob’s qubit | Bob’s correction |
|:---:|:---:|:---:|
| `00` | $a \| 0 \rangle + b \| 1 \rangle$ | none |
| `01` | $a \| 1 \rangle + b \| 0 \rangle$  | $X$ |
| `10` | $a \| 0 \rangle - b \| 1 \rangle$  | $Z$ |
| `11` | $a \| 1 \rangle - b \| 0 \rangle$| $X$ and $Z$ |

In the circuit, we perform these classically controlled operations via `if_test` statements. 

In [ ]:
# Bob applies corrections according to Alice's measurement outcomes

# Your turn: if the result from q1 is 1, apply ... to Bob's qubit; if the result from q2 is 1, apply ... to Bob's qubit

with qc.if_test((alice_bits[0], 1)):
   ## Apply the appropriate gate


qc.draw(output="mpl")

Bob’s qubit $q_0$ should now be in the original state $|\psi\rangle$, check it! Notice that Bob did not need to know $a$ or $b$: his correction depended only on Alice’s two classical bits.